# مشروع الدبلجة العربية — Course Translation V2.6 | Dynamic Glossary + Educational QA (Colab)

**نطاق البداية:** كورسات تعليمية إنجليزية بمتحدث واحد؛ التصميم العام للمصطلحات والسياق صالح لاختصاصات الكورسات المختلفة.

**الإصدار:** `V2.6` إصدار تطوير لدفتر الترجمة؛ لا يعني اعتماد مسودة الترجمة `V2.5`.

**المراجع:** Figma، الأقسام `02.9` إلى `02.27`، وخاصة Dynamic Glossary. هذا Notebook جديد ومرافق، وليس استبدالاً للـWhisperX الأصلي.

**المراحل التي لن تعمل تلقائياً:** WhisperX، إعادة الترجمة بواسطة LLM، توليد TTS، الدمج مع الفيديو. كل الملفات الناتجة مشتقة، ولا يُعدل ASR أو توقيت الكلمات أو `final_dub_ar`.

## 1 — إعداد الاتصال بملفات المشروع

تتطلب هذه الخلية Colab وDrive فقط؛ لا تستخدم GPU أو مفاتيح API.

In [ ]:
from pathlib import Path
from google.colab import drive
import json, hashlib, sys, subprocess, zipfile, shutil

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/ArabicDubbingV2')
SOURCE = PROJECT / 'source'
OUTPUTS = PROJECT / 'outputs'
SOURCE.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

# ضع ZIP المرفق (ArabicDubbing_TranslationV2_Code.zip) داخل مجلد source مرة واحدة.
CODE_ZIP = SOURCE / 'ArabicDubbing_TranslationV2_Code.zip'
if not CODE_ZIP.is_file():
    print('الكود غير موجود على Drive؛ ارفع ZIP إلى:', CODE_ZIP)
    from google.colab import files
    uploaded = files.upload()
    if 'ArabicDubbing_TranslationV2_Code.zip' not in uploaded:
        raise FileNotFoundError('المطلوب: ArabicDubbing_TranslationV2_Code.zip')
    CODE_ZIP.write_bytes(uploaded['ArabicDubbing_TranslationV2_Code.zip'])

CODE_DIR = Path('/content/ArabicDubbing_TranslationV2_Code')
CODE_DIR.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    for info in z.infolist():
        path=Path(info.filename)
        if path.is_absolute() or '..' in path.parts:
            raise ValueError('ZIP يحتوي مساراً غير آمن')
    z.extractall(CODE_DIR)
if not (CODE_DIR / 'arabic_dubbing_v2' / 'workflow.py').exists():
    raise FileNotFoundError('الحزمة غير مكتملة')
if str(CODE_DIR) not in sys.path:sys.path.insert(0,str(CODE_DIR))
print('Code installed locally for this Colab session:', CODE_DIR)
print('No translation or TTS executed.')

## 2 — اختبار الكود دون تشغيل النموذج أو استهلاك GPU

In [ ]:
checks = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                         cwd=CODE_DIR, capture_output=True, text=True)
print(checks.stdout)
print(checks.stderr[-10000:])
if checks.returncode != 0:raise RuntimeError('Failed offline tests; stop before processing any project Master.')
print('Offline tests passed. This is not a substitute for real course validation.')

## 3 — تحديد Master الحقيقي بلا افتراض أنه نسخة قديمة

بحسب سجل Figma توجد مسودة `V2.5` ولكنها لم تكن مرفوعة إلى Drive وقت توثيقها. الخلية تعرض الموجود ثم تتوقف إن لم تجد النسخة المحددة؛ **لن ترجع تلقائياً لإصدار V2.1**.

استعمل نفس مجلد الدرس الموجود أو حدّد مسار ملف من أي كورس. لا حاجة لتغيير المصطلحات يدوياً.

In [ ]:
# اختر ملف V2.5 إذا رفعته إلى Drive؛ غير الاسم يدوياً فقط بعد معرفة حالة نسختك.
MASTER_FILENAME = 'translation_master_v2_5_editorial_candidate.json'
# مثال: RUN_DIR = OUTPUTS / 'lesson'، أو اختر كورساً آخر.
RUN_DIR = OUTPUTS / 'lesson'
EXPLICIT_MASTER_PATH = None  # مثال: '/content/drive/MyDrive/.../translation_master_v2_5_editorial_candidate.json'

# معلومات اختيارية حقيقية؛ اتركها None إن لم تعرفها. لا تكتب قاموس مصطلحات ثابت.
COURSE_METADATA_OVERRIDES = {
    'id': None, 'title': None, 'description': None,
    'chapter': None, 'lesson': None, 'topic': None,
}

available = sorted(OUTPUTS.rglob('translation_master*.json'))
print('Masters discovered in Drive:')
for f in available[:80]:print(' -',f)
MASTER_PATH = Path(EXPLICIT_MASTER_PATH) if EXPLICIT_MASTER_PATH else RUN_DIR / MASTER_FILENAME
if not MASTER_PATH.is_file():
    raise FileNotFoundError('الـMaster المطلوب غير موجود. لا نستخدم نسخة أقدم تلقائياً: '+str(MASTER_PATH))
print('Selected master:',MASTER_PATH)
print('SHA256:',hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest())

## 4 — Data Contract + Context Builder + Dynamic Glossary + Educational QA

يستخرج مؤشرات المجال من بيانات الكورس أو مصطلحات متكررة فيه، ثم يحفظها **كمرشحات غير معتمدة**. جميع فحوص المعنى غير المنجزة تبقى `unknown`، ولا يتغير المصدر.

In [ ]:
from arabic_dubbing_v2.workflow import prepare_project
from arabic_dubbing_v2.common import fingerprint

src_bytes=MASTER_PATH.read_bytes()
src_hash=hashlib.sha256(src_bytes).hexdigest()
# نسخة جديدة باسم مرتبط ببصمة المصدر؛ لا تُستبدل إن كانت موجودة بالفعل.
OUT_DIR=RUN_DIR/'translation_v2_figma_implementation'
OUT_DIR.mkdir(parents=True,exist_ok=True)
DERIVED_MASTER = OUT_DIR / f'translation_master_v2_6_derived_{src_hash[:12]}.json'
if DERIVED_MASTER.exists():
    raise FileExistsError(f'هذه النسخة المشتقة موجودة مسبقاً، ولن أستبدلها: {DERIVED_MASTER}')
prepared = prepare_project(MASTER_PATH,DERIVED_MASTER,course_overrides=COURSE_METADATA_OVERRIDES)
assert hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest() == src_hash
assert all((u.get('analysis_v2') or {}).get('review_gate')=='pending_human_bilingual_review'
           for u in prepared['translation_units'])
print('Master derived:',DERIVED_MASTER)
print('Report:',json.dumps(prepared['translation_v2_report'],ensure_ascii=False,indent=2)[:12000])
print('Domain hint:',prepared['terminology_v2']['domain_analysis'])
print('Candidate terms:',len(prepared['terminology_v2']['candidates']))
print('No model calls; original translation texts, ASR, word refs and timestamps remain unchanged.')

## 5 — استخراج ملفات خطط ترجمة ومراجعة المصطلحات (لا LLM هنا)

يدعم هذا القسم أربع مراحل Prompt منفصلة: accurate → natural → timing_candidate → semantic QA، ويضيف خطط Contextual Validation للمصطلحات المكتشفة. **لن ينفّذ هذه المطالبات أو يستهلك API.**

In [ ]:
from arabic_dubbing_v2.common import save_new_json
from arabic_dubbing_v2.translation import build_all_pass_plans
from arabic_dubbing_v2.terminology import validation_plans

PLAN_TAG = src_hash[:12]
TRANSLATION_PLANS=OUT_DIR / f'translation_pass_plans_{PLAN_TAG}.json'
GLOSSARY_PLANS=OUT_DIR / f'dynamic_glossary_validation_plans_{PLAN_TAG}.json'
translation_plan=build_all_pass_plans(prepared)
glossary_plan=validation_plans(prepared['terminology_v2'],prepared['translation_v2_metadata']['course'])
save_new_json(TRANSLATION_PLANS,translation_plan)
save_new_json(GLOSSARY_PLANS,glossary_plan)
print('TU plans:',len(translation_plan['plans']), '| Dynamic glossary plans:',len(glossary_plan['plans']))
print('Saved:',TRANSLATION_PLANS,GLOSSARY_PLANS)
print('NO LLM REQUESTS WERE MADE.')

## 6 — تشغيل الترجمة الجديدة اختيارياً فقط بعد موافقة صريحة

عندما تختار مزوّداً (يمكن أن يكون **محلياً ومجانياً**) وتضيف وظيفة `TRANSLATION_BACKEND` التي تُرجع JSON لكل مرحلة، يمكنك تفعيل العلم بالأسفل. لا يوجد أي مزوّد افتراضي أو اتصال مدفوع مضمن. كل الناتج **مسودة جديدة** منفصلة عن الـMaster؛ ولا يُكتب `final_dub_ar`.

In [ ]:
from arabic_dubbing_v2.translation import generate_draft
ENABLE_NEW_TRANSLATION = False
TRANSLATION_BACKEND = None  # Callable(instruction, inputs, prompt_version) -> parsed dict
SELECTED_UNIT_IDS_FOR_GENERATION = []  # اختَر المعرفات بعد مراجعة ملفات الخطط

if not ENABLE_NEW_TRANSLATION:
    print('Generation disabled. Existing translations preserved.')
else:
    if TRANSLATION_BACKEND is None:raise RuntimeError('لا يوجد Backend مترجم مختار؛ لا طلب API.')
    if not SELECTED_UNIT_IDS_FOR_GENERATION:raise ValueError('حدد وحدات للتجربة فقط.')
    for unit_id in SELECTED_UNIT_IDS_FOR_GENERATION:
        result = generate_draft(prepared,unit_id,TRANSLATION_BACKEND,allow_translation=True)
        out = OUT_DIR / f'candidate_generated_{unit_id}_{PLAN_TAG}.json'
        save_new_json(out,result)
        print('Unapproved draft:',out)


## 7 — التخطيط للمعاينة الصوتية دون توليد فعلي

Edge TTS يتصل بخدمة إنترنت خارجية مجانية للاختبار، لا Google TTS. التجربة لا تعني الموافقة على النص أو الصوت. لا يختار معرفات محددة مرتبطة بدورة واحدة، ويقرأ من `natural_ar` فقط.

In [ ]:
from arabic_dubbing_v2.speech import run_edge_previews
RUN_TTS_PREVIEW = False
ALLOW_ONLINE_TTS = False
PREVIEW_UNIT_IDS = None  # None = كل الوحدات التي تمتلك natural_ar
VOICE = 'ar-SA-HamedNeural'
RATE = '+0%'

if RUN_TTS_PREVIEW:
    import importlib.util
    if importlib.util.find_spec('edge_tts') is None:
        raise RuntimeError('ثبّت edge-tts اختيارياً عند الموافقة على المعاينة؛ لم نثبته تلقائياً.')
preview=run_edge_previews(prepared, OUT_DIR/'temporary_audio_previews',
                          unit_ids=PREVIEW_UNIT_IDS,voice=VOICE,rate=RATE,
                          execute=RUN_TTS_PREVIEW,allow_online_tts=ALLOW_ONLINE_TTS)
print(preview['status'])
print(json.dumps(preview.get('plan',preview.get('results')),ensure_ascii=False,indent=2)[:9000])

## 8 — خيار منفصل: تجربة Builder عام من RAW WhisperX (لا يغيّر V1.1)

**لا تشغّله على ملف الماستر الذي أُصلحت وحداته بشرياً** إلا إذا أردت تجربة مقارنة مستقلة. يحافظ على كل الكلمات المحاذاة. قد تختلف حدود وحداته عن Builder V1.1، ولذلك لا يستبدل النسخة المراجعة.

In [ ]:
EXPERIMENT_WITH_NEW_BUILDER = False
ALIGNED_RAW_PATH = RUN_DIR / 'whisperx_aligned_raw.json'
if not EXPERIMENT_WITH_NEW_BUILDER:
    print('Experimental Builder disabled; saved reviewed TUs remain unchanged.')
else:
    from arabic_dubbing_v2.boundaries import build_units_from_whisperx
    from arabic_dubbing_v2.common import read_json, save_new_json
    raw=read_json(ALIGNED_RAW_PATH)
    source_segs=raw.get('source_segments',raw.get('segments'))
    comparison=build_units_from_whisperx(source_segs,
                       course=prepared['translation_v2_metadata']['course'])
    dest=OUT_DIR / f'experimental_builder_v2_{PLAN_TAG}.json'
    save_new_json(dest,comparison)
    print('Comparison only:',dest,comparison['builder_report'])


## الخلاصة

هنا أصبح لدينا Master مشتق، سياق بجوار الوحدة، كشف مجال تمهيدي، استخراج مصطلحات ديناميكي مع ذاكرة موافقات خاصة بالكورس، خطط ترجمة ومراجعة، بوابة QA لا تمنح موافقة وهمية، ومعاينة صوتية اختيارية. **لم تُعَد الترجمة أو الصوت تلقائياً**. المزامنة والصوت النهائي يخضعان لمراجعة منفصلة لاحقاً.